# Fine-tune MMS TTS on Kikuyu Voice Data

This notebook fine-tunes `facebook/mms-tts-kik` on your prepared Kikuyu dataset.

## Before running:
1. Upload `dataset/prepared/` folder to your Google Drive as `MyDrive/kikuyu-tts/prepared/`
2. Set your HuggingFace token below
3. Runtime → Change runtime type → GPU (T4)
4. Run all cells top to bottom

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers==4.40.0 accelerate datasets soundfile librosa resampy
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
print('✓ Dependencies installed')

In [ ]:
# ── Step 2: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DATASET_DIR = '/content/drive/MyDrive/kikuyu-tts/prepared'
assert os.path.exists(DATASET_DIR), f'Dataset not found at {DATASET_DIR}'
print(f'✓ Dataset found: {DATASET_DIR}')

In [ ]:
# ── Step 3: HuggingFace login ─────────────────────────────────────────────────
# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN      = 'hf_YOUR_TOKEN_HERE'          # ← replace this
HF_REPO_NAME  = 'kikuyu-tts-custom'            # ← your model name on HuggingFace
HF_USERNAME   = 'your-hf-username'             # ← your HuggingFace username

from huggingface_hub import login
login(token=HF_TOKEN)
print(f'✓ Logged in — model will be saved as {HF_USERNAME}/{HF_REPO_NAME}')

In [ ]:
# ── Step 4: Load dataset ──────────────────────────────────────────────────────
import csv
import soundfile as sf
import numpy as np
from pathlib import Path

def load_metadata(csv_path):
    rows = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            wav_path = os.path.join(DATASET_DIR, 'wavs', row['filename'])
            if os.path.exists(wav_path):
                rows.append({'audio_path': wav_path, 'text': row['text']})
    return rows

train_data = load_metadata(os.path.join(DATASET_DIR, 'train.csv'))
val_data   = load_metadata(os.path.join(DATASET_DIR, 'val.csv'))

print(f'✓ Train: {len(train_data)} samples')
print(f'✓ Val:   {len(val_data)} samples')
print(f'  Sample: {train_data[0]}')

In [ ]:
# ── Step 5: Load base model ───────────────────────────────────────────────────
import torch
from transformers import VitsModel, AutoTokenizer

MODEL_ID  = 'facebook/mms-tts-kik'
device    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading {MODEL_ID} on {device}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = VitsModel.from_pretrained(MODEL_ID)
model     = model.to(device)
model.train()

print(f'✓ Model loaded — {sum(p.numel() for p in model.parameters()):,} parameters')

In [ ]:
# ── Step 6: Dataset class ─────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class KikuyuTTSDataset(Dataset):
    def __init__(self, data, tokenizer, target_sr=22050):
        self.data      = data
        self.tokenizer = tokenizer
        self.sr        = target_sr

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row  = self.data[idx]
        wav, sr = sf.read(row['audio_path'], dtype='float32', always_2d=False)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        wav  = torch.tensor(wav, dtype=torch.float32)

        tokens = self.tokenizer(
            row['text'],
            return_tensors='pt',
            padding=False
        )
        return {
            'input_ids':      tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'waveform':       wav,
        }

train_dataset = KikuyuTTSDataset(train_data, tokenizer)
val_dataset   = KikuyuTTSDataset(val_data,   tokenizer)
print(f'✓ Datasets ready')

In [ ]:
# ── Step 7: Training config ───────────────────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

NUM_EPOCHS   = 50     # increase to 100–200 for better quality with more data
BATCH_SIZE   = 1      # VITS needs batch_size=1 due to variable-length waveforms
LR           = 2e-5   # low LR for fine-tuning (not training from scratch)
SAVE_EVERY   = 10     # save checkpoint every N epochs
OUTPUT_DIR   = '/content/drive/MyDrive/kikuyu-tts/checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f'✓ Training for {NUM_EPOCHS} epochs, LR={LR}')

In [ ]:
# ── Step 8: Training loop ─────────────────────────────────────────────────────
from tqdm.notebook import tqdm

best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for sample in tqdm(train_dataset, desc=f'Epoch {epoch}/{NUM_EPOCHS}', leave=False):
        input_ids      = sample['input_ids'].unsqueeze(0).to(device)
        attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
        target_wav     = sample['waveform'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred_wav = outputs.waveform.squeeze()

        # Align lengths for L1 loss
        min_len = min(pred_wav.shape[-1], target_wav.shape[-1])
        loss    = torch.nn.functional.l1_loss(pred_wav[..., :min_len], target_wav[:min_len])

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()

    scheduler.step()
    avg_train = train_loss / len(train_dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for sample in val_dataset:
            input_ids      = sample['input_ids'].unsqueeze(0).to(device)
            attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
            target_wav     = sample['waveform'].to(device)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            pred_wav       = outputs.waveform.squeeze()
            min_len        = min(pred_wav.shape[-1], target_wav.shape[-1])
            val_loss      += torch.nn.functional.l1_loss(pred_wav[..., :min_len], target_wav[:min_len]).item()

    avg_val = val_loss / max(1, len(val_dataset))
    print(f'Epoch {epoch:3d} | Train: {avg_train:.5f} | Val: {avg_val:.5f}')

    # Save best
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(f'{OUTPUT_DIR}/best')
        tokenizer.save_pretrained(f'{OUTPUT_DIR}/best')
        print(f'  ✓ Best model saved (val_loss={avg_val:.5f})')

    # Periodic checkpoint
    if epoch % SAVE_EVERY == 0:
        model.save_pretrained(f'{OUTPUT_DIR}/epoch_{epoch}')
        tokenizer.save_pretrained(f'{OUTPUT_DIR}/epoch_{epoch}')

print('\n✓ Training complete!')

In [ ]:
# ── Step 9: Test the fine-tuned model ─────────────────────────────────────────
from IPython.display import Audio, display
import scipy

model.eval()

test_sentences = [
    'Wĩ mwega?',
    'Nĩ ngatho muno muno',
    'Ngai akuhe gĩthomo na ũhoro mwega',
]

for text in test_sentences:
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model(**inputs).waveform
    wav = output[0].cpu().numpy()
    sr  = model.config.sampling_rate

    print(f'\n"{text}"')
    display(Audio(wav, rate=sr))

    # Save sample
    scipy.io.wavfile.write(f'/content/test_{test_sentences.index(text)}.wav', sr, wav)

In [ ]:
# ── Step 10: Push to HuggingFace Hub ──────────────────────────────────────────
# This makes the model available for your mms-server

best_model_dir = f'{OUTPUT_DIR}/best'

# Load best checkpoint
from transformers import VitsModel, AutoTokenizer
best_model     = VitsModel.from_pretrained(best_model_dir)
best_tokenizer = AutoTokenizer.from_pretrained(best_model_dir)

repo_id = f'{HF_USERNAME}/{HF_REPO_NAME}'
best_model.push_to_hub(repo_id, token=HF_TOKEN)
best_tokenizer.push_to_hub(repo_id, token=HF_TOKEN)

print(f'\n✓ Model pushed to: https://huggingface.co/{repo_id}')
print(f'\nTo use in your mms-server, update main.py:')
print(f'  MODEL_ID = "{repo_id}"')